In [ ]:
from aligner.networksHandling import helper as helper_networks
from aligner.poissonAlignment import helper as helper_poisson
from aligner.montecarlo import helper as helper_montecarlo
from aligner.analysis import helper as helper_analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import pickle
import seaborn as sns

In [ ]:
#create directories to store results (if they don't exist)
!mkdir -p results/

We will create four synthetic networks using connectome A.2 as the blueprint. We define a temporal scaling parameter $\alpha_t$ for each network. However, instead of assigning the same scaling factor to every edge, each edge is given its own scaling parameter $\alpha_{i,j}^{t}$, thereby violating our assumption that all edges share a common temporal scaling.

Each $\alpha_{i,j}^{t}$ is sampled independently from a Gaussian distribution:

$$
\alpha_{i,j}^{t} \sim \mathcal{N}(\alpha_t, \sigma).
$$

We then perform a series of experiments by varying the value of $\sigma$, ranging from low variance (where edge-specific scaling parameters remain close to $\alpha_t$) to high variance (corresponding to increasingly strong violations of our assumption).

Since $\alpha_{i,j}^{t}$ can't be negative, we can replace negative values with zeros (i.e., rectified Gaussian).

In [ ]:
#blueprint from which we'll generate the synthetic networks:
connectome_A2 = helper_networks.load_networks(data_folder="data/processed/celegans/", net_ids=[8])[0] #last connectome
connectome_A2

In [ ]:
#alpha_t and sigmas:
sigma_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6]
alpha_t_vector = [0.25,0.5,0.75,1.0]

In [ ]:
#synthesize networks:
def synthesize_networks(blueprint, alpha_t_vector, sigma_gaussian, seed=None):
    if seed is not None:
        np.random.seed(seed)

    N = blueprint.shape[0]
    T = len(alpha_t_vector)
    synthetic_nets = np.zeros((T, N, N))

    alpha_t_ij_all_vectors = []

    for t in range(T):
        alpha_t = alpha_t_vector[t]
        alpha_t_ij_vector = []

        for i in range(N):
            for j in range(N):
                alpha_t_ij = max(np.random.normal(alpha_t, sigma_gaussian),0) 
                alpha_t_ij_vector.append(alpha_t_ij)
                poisson_rate = blueprint[i, j] * alpha_t_ij
                synthetic_nets[t, i, j] = np.random.poisson(poisson_rate)
        alpha_t_ij_all_vectors.append(alpha_t_ij_vector)
        
    return synthetic_nets, alpha_t_ij_all_vectors

In [ ]:
#Groups (to constrain the alignment):
neuron_metadata = pd.read_csv("data/processed/celegans/neuron_metadata.csv", index_col=0) 
groups = np.array(neuron_metadata["location"])
print(groups)

In [ ]:
def count_edge_trends(A):
    """
    Count monotonically increasing/decreasing edges and edges without
    a monotonic trend, excluding edges that are always zero.

    A has shape (T, N, N).
    """
    A = np.asarray(A)

    if A.ndim != 3:
        raise ValueError("A must have shape (T, N, N)")

    # Edges that are active at least once
    active_mask = np.any(A != 0, axis=0)

    # Differences between consecutive time points
    diff = np.diff(A, axis=0)

    # Monotonic direction
    increasing_mask = np.all(diff >= 0, axis=0)
    decreasing_mask = np.all(diff <= 0, axis=0)

    # Must actually change at least once
    has_increase = np.any(diff > 0, axis=0)
    has_decrease = np.any(diff < 0, axis=0)

    # Strictly increasing/decreasing edges
    increasing_mask &= has_increase & active_mask
    decreasing_mask &= has_decrease & active_mask

    n_increasing = np.sum(increasing_mask)
    n_decreasing = np.sum(decreasing_mask)

    # Active edges that are neither strictly increasing nor decreasing
    n_notrend = np.sum(
        active_mask & ~(increasing_mask | decreasing_mask)
    )

    return n_increasing, n_decreasing, n_notrend

In [ ]:
# EXPERIMENTS .....................................
import numpy as np
import matplotlib.pyplot as plt
import pickle
import time
from scipy.stats import pearsonr

alpha_t_vector = [0.25, 0.5, 0.75, 1.0]
R = 5  # Number of Monte Carlo runs per experiment

# Summary dictionary
dict_res_sigmas = {"sigmas": sigma_values}

alphas_for_dict = []
percincr_for_dict = []
percdecr_for_dict = []
percnotrend_for_dict = []
corr_for_dict = []
std_corr_for_dict = []
As_for_dict = []

for sigma in sigma_values:

    print("*******************************************")
    print(f"Experiment with sigma = {sigma}")

    # --------------------------------------------------
    # 1. SYNTHESIZE NETWORKS
    # --------------------------------------------------
    A, all_alphatij = synthesize_networks(
        blueprint=connectome_A2,
        alpha_t_vector=alpha_t_vector,
        sigma_gaussian=sigma,
        seed=99
    )

    T, N, _ = A.shape

    As_for_dict.append(A)

    # --------------------------------------------------
    # 2. ANALYZE SYNTHESIZED NETWORKS
    # --------------------------------------------------

    # Distribution of alpha_tij
    for t in range(T):
        plt.hist(
            all_alphatij[t],
            alpha=0.65,
            label=f"alpha_t={alpha_t_vector[t]}",
            bins=100
        )

    plt.title(f"sigma={sigma} ---- alpha^t_ij distribution")
    plt.legend(loc="upper right")

    ax = plt.gca()
    for spine in ["right", "top"]:
        ax.spines[spine].set_visible(False)

    plt.show()

    # Computed alpha_t
    computed_alpha = helper_poisson.get_alpha(
        A, helper_poisson.get_ST(A)
    )
    print(
        "The computed alpha_t for these synthetic networks are =",
        computed_alpha
    )

    # Alpha distribution for summary dictionary
    alphas_for_dict.append(all_alphatij)

    # Percentage of nonzero and zero edges
    num_nonzero_edges = np.sum(np.sum(A, axis=0) > 0)
    num_zero_edges = np.sum(np.sum(A, axis=0) == 0)

    print("% non-zero edges:", 100 * num_nonzero_edges / (N * N))
    print("% zero edges:", 100 * num_zero_edges / (N * N))

    # Edge trends
    num_increasing, num_decreasing, num_notrend = count_edge_trends(A)

    if num_nonzero_edges > 0:
        perc_increasing = 100 * num_increasing / num_nonzero_edges
        perc_decreasing = 100 * num_decreasing / num_nonzero_edges
        perc_notrend = 100 * num_notrend / num_nonzero_edges
    else:
        perc_increasing = np.nan
        perc_decreasing = np.nan
        perc_notrend = np.nan

    print("Increasing vs decreasing:", perc_increasing, perc_decreasing)

    percincr_for_dict.append(perc_increasing)
    percdecr_for_dict.append(perc_decreasing)
    percnotrend_for_dict.append(perc_notrend)

    # Correlation between networks
    print("Network correlation")

    corr = np.zeros((T, T))

    for i in range(T):
        for j in range(T):
            w_i = A[i].flatten()
            w_j = A[j].flatten()

            corr[i, j] = pearsonr(w_i, w_j).statistic

    off_diagonal = corr[~np.eye(T, dtype=bool)]
    mean_corr = off_diagonal.mean()
    std_corr = np.std(off_diagonal)

    print("Mean correlation between networks:", mean_corr)

    corr_for_dict.append(mean_corr)
    std_corr_for_dict.append(std_corr)

    # --------------------------------------------------
    # 3. ALIGNMENT AND MONTE CARLO SAMPLING
    # --------------------------------------------------

    # Initial permutations (preserving left/right groups)
    P = helper_montecarlo.initial_permutation_groups(A, groups)

    # Ground-truth permutation
    gtP = [np.arange(N) for _ in range(T)]

    # Inverse temperature values
    beta_spacing = 1.005
    betas = np.array([beta_spacing**n for n in range(-12, 1)])
    B = len(betas)

    # Ground-truth energy
    Egt = helper_poisson.calculateEnergy_total(
        A,
        gtP,
        helper_poisson.get_Salpha(A),
        N,
        T
    )

    # --------------------------------------------------
    # 4. MONTE CARLO RUNS
    # --------------------------------------------------

    all_results = []

    for run in range(R):

        start = time.process_time()

        Esweeps_r, samples, groundState, MCacceptance_rate = (
            helper_montecarlo.sampling_poisson_groups(
                A, P, betas, 10000, 5000, 2, groups, ""
            )
        )

        end = time.process_time()

        print(f"Process time (run {run}, s): {end - start}")

        # Save only Monte Carlo results and alpha distributions.
        # The synthesized matrix A is NOT saved.
        results = {
            "E": Esweeps_r,
            "E_GT": Egt,
            "GS": groundState,
            "Samples": samples,
            "alphadistr": all_alphatij,
            "MCacceptance_rate": MCacceptance_rate,
            "sigma": sigma,
            "run": run
        }

        with open(f"results/robustness_sigma{sigma}_run{run}.pkl", "wb") as file:
            pickle.dump(results, file)

        all_results.append(results)

        # Plot energy sweeps
        plt.figure(figsize=(4, 2))

        for b in range(B):
            plt.plot(Esweeps_r[b], label=round(betas[b], 3))

        plt.plot(
            [Egt for _ in range(len(Esweeps_r[0]))],
            color="red",
            linestyle="dashed",
            label="GT Energy"
        )

        plt.title(f"sigma={sigma}, run={run}")
        plt.show()

# --------------------------------------------------
# 5. COLLECT SUMMARY STATISTICS
# --------------------------------------------------

dict_res_sigmas["alphadistr"] = alphas_for_dict
dict_res_sigmas["perc_increasing"] = percincr_for_dict
dict_res_sigmas["perc_decreasing"] = percdecr_for_dict
dict_res_sigmas["perc_notrend"] = percnotrend_for_dict
dict_res_sigmas["mean_correlation"] = corr_for_dict
dict_res_sigmas["std_correlation"] = std_corr_for_dict
dict_res_sigmas["A"] = As_for_dict

print("All experiments completed.")

In [ ]:
# ANALYSIS.................................................................................
accuracies = []
num_steps = []

for sigma in sigma_values:

    print("----------------------------------------------------")
    print(f"Experiment with SIGMA = {sigma}")
    file_results = f"results/robustness_sigma{sigma}_" 

    A = dict_res_sigmas["A"][dict_res_sigmas["sigmas"].index(sigma)]
    
    for run in range(R):
        plt.figure(figsize=(3,1))
        with open(f"{file_results}run{run}.pkl", "rb") as file:
            results = pickle.load(file)
            all_results.append(results)

        for b in range(B):
            plt.plot(results["E"][b], label=round(betas[b], 3))
        plt.plot([results["E_GT"] for _ in range(len(results["E"][0]))], color="red", linestyle="dashed", label = "Ground truth")

        plt.show()

    #sampling average:
    S = len(results["Samples"])
    samples = np.zeros((S*R, T, N)) 
    for run in range(R):
        run_samples = all_results[run]["Samples"]
        samples[run*S:(run+1)*S, :, :] = run_samples  

    ensemble = samples
    ensemble_node_mapping = helper_analysis.infer_node_mapping(ensemble)
    ensemble_node_mapping = helper_analysis.mark_empty_nodes(ensemble_node_mapping, A)
    ensemble_node_mapping = helper_analysis.order_node_mapping(ensemble_node_mapping, A, groups) #to sort nodes by desc strngth in the densest snapshot (in this case, A.2)
    
    accuracy = helper_analysis.compute_accuracy(ensemble_node_mapping) #disregarding non-alignable nodes
    helper_analysis.show_alignment(ensemble_node_mapping, N, T, True, True, [1,2,3,4], 
                                   f"Inferred node mapping (accuracy={100*accuracy:.2f})", groups)
    plt.show()

    accuracies.append(accuracy)

In [ ]:
# PLOT ..........................................................
from matplotlib.gridspec import GridSpec

sns.set(style="ticks")
plt.rcParams["font.family"] = "Arial"

font_scale = 2.1
size_label, size_values = 14 * font_scale, 12 * font_scale
size_title, size_abc = 16 * font_scale, 23 * font_scale

fig = plt.figure(figsize=(40, 20))
gs = GridSpec(3, 4, figure=fig, hspace=0.7, height_ratios=[0.5,0.25,0.25])

ax_a, ax_b, ax_c, ax_d = [fig.add_subplot(gs[0, i*1:(i+1)*1]) for i in range(4)]
ax_e = fig.add_subplot(gs[1:3, 0:2])
ax_f = fig.add_subplot(gs[1, 2:4])
ax_g = fig.add_subplot(gs[2, 2:4])

# a-d: alpha distributions
v_colors = plt.cm.viridis(np.linspace(0, 1, 4))
sigmas_for_distr = [0.0, 0.2, 0.8, 1.6]

for k, ax in enumerate([ax_a, ax_b, ax_c, ax_d]):
    sigma = sigmas_for_distr[k]
    idx = dict_res_sigmas["sigmas"].index(sigma)
    distribution = dict_res_sigmas["alphadistr"][idx]

    for t in range(T):
        ax.hist(
            distribution[t],
            bins=50,
            alpha=0.7,
            color=v_colors[3-t],
            label=rf"$t={t+1}$, $\alpha_t={alpha_t_vector[t]}$"
        )

    ax.set_title(rf"$\sigma={sigma}$", fontsize=size_title, pad=18)
    ax.set_xlabel(
        r"Edge-dev. parameter, $\alpha^t_{i,j}$",
        fontsize=size_label, labelpad=18
    )

    if k == 0:
        ax.set_ylabel("Number of edges", fontsize=size_label, labelpad=18)
    if k == 3:
        ax.legend(loc="upper right", frameon=False, fontsize=size_values)

    ax.set_xlim(left=-0.1)
    ax.tick_params(axis="both", labelsize=size_values)
    ax.spines[["right", "top"]].set_visible(False)
    
# e: perc of increasing edges (i.e., following the original alpha_t trend)
normal = "#c4c4c4"
darker = "#a19f9f"
highlight = {0.0, 0.2, 0.8, 1.6}
point_colors = [darker if s in highlight else normal for s in sigma_values]
ax_e.plot(sigma_values, dict_res_sigmas["perc_notrend"], "o-", label="no trend", 
          linewidth=4, color=normal)
ax_e.scatter(sigma_values, dict_res_sigmas["perc_notrend"], 
            s=100, color=point_colors, zorder=3)

normal = "#e0a5fa"
darker = "#b683cc"
highlight = {}
point_colors = [darker if s in highlight else normal for s in sigma_values]
ax_e.plot(sigma_values, dict_res_sigmas["perc_increasing"], "o-", label="increasing edges", 
          linewidth=4, color=normal)
ax_e.scatter(sigma_values, dict_res_sigmas["perc_increasing"], 
            s=200, color=point_colors, zorder=3)

normal = "#d5d973"
darker = "#a5a859"
highlight = {}
point_colors = [darker if s in highlight else normal for s in sigma_values]
ax_e.plot(sigma_values, dict_res_sigmas["perc_decreasing"], "o-", label="decreasing edges", 
          linewidth=4, color=normal)
ax_e.scatter(sigma_values, dict_res_sigmas["perc_decreasing"], 
            s=200, color=point_colors, zorder=3)

ax_e.set_xlabel(
    r"Standard deviation, $\sigma$",
    fontsize=size_label, labelpad=18
)
ax_e.set_ylabel("Percentage of active edges (%)", fontsize=size_label, labelpad=18)
ax_e.legend(loc="center right", frameon=False, fontsize=size_values)
ax_e.spines[["right", "top"]].set_visible(False)
ax_e.tick_params(axis="both", labelsize=size_values)
ax_e.set_xticks(sigma_values)
# ax_e.set_ylim(0, 42)

# f: correlation (networks) vs sigma

normal = "#9b9c9a"
darker = "#757575"
highlight = {}

means = np.array(dict_res_sigmas["mean_correlation"])
stds = np.array(dict_res_sigmas["std_correlation"])

point_colors = [darker if s in highlight else normal for s in sigma_values]

# Mean correlation and connecting line
ax_f.plot(
    sigma_values,
    means,
    color=normal,
    linewidth=3
)
ax_f.plot(
    sigma_values,
    [0.5] * len(sigma_values),
    linestyle="dashed",
    color="#b8b8b8",
    zorder=0
)
# Error bars: mean ± standard deviation
ax_f.errorbar(
    sigma_values,
    means,
    yerr=stds,
    fmt="none",
    ecolor=normal,
    elinewidth=2.5,
    capsize=8,
    capthick=2.5,
    zorder=2
)

# Highlighted data points
ax_f.scatter(
    sigma_values,
    means,
    color=point_colors,
    s=200,
    zorder=3
)

ax_f.set_xticks(sigma_values)
ax_f.set_ylabel("Mean network\ncorrelation", fontsize=size_label, labelpad=0)
ax_f.tick_params(axis="both", labelsize=size_values)
ax_f.spines[["right", "top"]].set_visible(False)
ax_f.set_ylim(0.0, 1.0)

# g: accuracy vs sigma
normal = "#9b9c9a"
darker = "#757575"
highlight = {}
point_colors = [darker if s in highlight else normal for s in sigma_values]

ax_g.plot(sigma_values, 100*np.array(accuracies), color=normal, linewidth=3)
ax_g.scatter(sigma_values, 100*np.array(accuracies), color=point_colors, s=200, zorder=3)
ax_g.set_xticks(sigma_values)
ax_g.set_xlabel(
    r"Standard deviation, $\sigma$",
    fontsize=size_label, labelpad=18
)
ax_g.set_ylabel("Accuracy (%)", fontsize=size_label, labelpad=0)
ax_g.set_ylim(0, 105)
ax_g.tick_params(axis="both", labelsize=size_values)
ax_g.spines[["right", "top"]].set_visible(False)

# panel labels
for ax, label in zip(
    [ax_a, ax_b, ax_c, ax_d, ax_e, ax_f, ax_g],
    "abcdefg"
):
    if label in "abcde":
        x, y = (-0.01, 1.15) 
    elif label in "fg": 
        x, y = (-0.01, 1.35) 
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=size_abc,
        fontweight="bold"
    )


plt.show()